# 🏀 Find Your Player's Twin: NBA Similarity Engine

> **Who is the modern Kobe? The next Magic?** Using box scores, play-type data,
> hustle metrics, and physical profiles, we build three different similarity engines
> and discover surprising player twins.

**Part 8 of 10** in the [wyattowalsh/basketball](https://www.kaggle.com/datasets/wyattowalsh/basketball) companion notebook series.

---

## What makes this notebook unique?

Most player comparison tools use basic box-score stats. The `wyattowalsh/basketball` dataset
lets us build similarity across **four dimensions** no other public dataset combines:

| Dimension | Features | Source |
|-----------|----------|--------|
| Statistical | PTS, REB, AST, efficiency, usage | `agg_player_season` |
| Play Style | Isolation, PnR, SpotUp, PostUp, Transition, Cut frequency | `fact_synergy` |
| Hustle | Contested shots, deflections, loose balls, charges, screens | `fact_player_game_hustle` |
| Physical | Height, weight, position | `dim_player` |

We implement **three similarity methods** (cosine, Euclidean, Gower) and show how
they produce different but complementary player comparisons.

---

## Table of Contents

1. [Setup & Data Loading](#1)
2. [Feature Engineering](#2)
3. [Similarity Methods](#3)
4. [Interactive Similarity Finder](#4)
5. [Cross-Era Comparison](#5)
6. [Similarity Network](#6)
7. [Draft Prospect Comparison](#7)
8. [Conclusion & Cross-Links](#8)

<a id="1"></a>
## 1. Setup & Data Loading

In [ ]:
!pip install -q "duckdb>=1.4,<2" "polars>=1.38,<2" "plotly>=5.18,<6" "scikit-learn>=1.4,<2"

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, OneHotEncoder

warnings.filterwarnings("ignore", category=FutureWarning)

# Import shared utilities
_kaggle_path = Path("/kaggle/input/basketball")
sys.path.insert(0, str(_kaggle_path if _kaggle_path.exists() else Path(".")))
from nbadb_utils import COLORS, TEMPLATE, takeaway, get_connection

conn = get_connection()

<a id="2"></a>
## 2. Feature Engineering

We build a rich player feature vector per season by joining four data sources:

- **Statistical profile** (12 features): scoring, rebounding, playmaking, efficiency
- **Play style** (6 features): possession fraction per play type from Synergy data
- **Hustle** (5 features): defensive effort metrics averaged per game
- **Physical** (3 features): height, weight, position

We filter to players with >= 30 games and >= 500 total minutes to exclude
small-sample outliers.

In [ ]:
# --- Build the player feature matrix ---

feature_sql = """
WITH hustle_season AS (
    SELECT
        h.player_id,
        g.season_year,
        COUNT(*) AS hustle_gp,
        SUM(h.contested_shots)::FLOAT / NULLIF(COUNT(*), 0)       AS contested_shots_pg,
        SUM(h.deflections)::FLOAT / NULLIF(COUNT(*), 0)           AS deflections_pg,
        SUM(h.loose_balls_recovered)::FLOAT / NULLIF(COUNT(*), 0) AS loose_balls_pg,
        SUM(h.charges_drawn)::FLOAT / NULLIF(COUNT(*), 0)         AS charges_drawn_pg,
        SUM(h.screen_assists)::FLOAT / NULLIF(COUNT(*), 0)        AS screen_assists_pg
    FROM fact_player_game_hustle h
    JOIN dim_game g ON h.game_id = g.game_id
    WHERE g.season_type = 'Regular Season'
    GROUP BY h.player_id, g.season_year
),

synergy_pivot AS (
    SELECT
        player_id,
        season_year,
        MAX(CASE WHEN play_type = 'Isolation' THEN poss_pct END)    AS iso_poss_frac,
        MAX(CASE WHEN play_type = 'PRBallHandler' THEN poss_pct END) AS pnr_bh_frac,
        MAX(CASE WHEN play_type = 'Spotup' THEN poss_pct END)       AS spotup_frac,
        MAX(CASE WHEN play_type = 'Postup' THEN poss_pct END)       AS postup_frac,
        MAX(CASE WHEN play_type = 'Transition' THEN poss_pct END)   AS transition_frac,
        MAX(CASE WHEN play_type = 'Cut' THEN poss_pct END)          AS cut_frac
    FROM fact_synergy
    WHERE entity_type = 'player'
      AND type_grouping = 'offensive'
    GROUP BY player_id, season_year
),

height_inches AS (
    -- dim_player.height is stored as e.g. '6-8'; convert to inches
    SELECT
        player_id,
        full_name,
        position,
        weight,
        CASE
            WHEN height LIKE '%-%'
            THEN CAST(SPLIT_PART(height, '-', 1) AS INT) * 12
                 + CAST(SPLIT_PART(height, '-', 2) AS INT)
            ELSE NULL
        END AS height_in
    FROM dim_player
    WHERE is_current = true
)

SELECT
    -- Identifiers
    aps.player_id,
    dp.full_name   AS player_name,
    aps.season_year,
    aps.gp,
    aps.total_min,

    -- Statistical profile (12 features)
    aps.avg_pts,
    aps.avg_reb,
    aps.avg_ast,
    aps.avg_stl,
    aps.avg_blk,
    aps.avg_tov,
    aps.avg_ts_pct,
    aps.avg_usg_pct,
    aps.avg_net_rating,
    aps.avg_pie,
    aps.fg3_pct,
    aps.ft_pct,

    -- Play style (6 features)
    sp.iso_poss_frac,
    sp.pnr_bh_frac,
    sp.spotup_frac,
    sp.postup_frac,
    sp.transition_frac,
    sp.cut_frac,

    -- Hustle (5 features)
    hs.contested_shots_pg,
    hs.deflections_pg,
    hs.loose_balls_pg,
    hs.charges_drawn_pg,
    hs.screen_assists_pg,

    -- Physical (3 features)
    dp.height_in,
    dp.weight,
    dp.position

FROM agg_player_season aps

JOIN height_inches dp
    ON aps.player_id = dp.player_id

LEFT JOIN hustle_season hs
    ON aps.player_id = hs.player_id
   AND aps.season_year = hs.season_year

LEFT JOIN synergy_pivot sp
    ON aps.player_id = sp.player_id
   AND aps.season_year = sp.season_year

WHERE aps.season_type = 'Regular Season'
  AND aps.gp >= 30
  AND aps.total_min >= 500
ORDER BY aps.season_year DESC, aps.avg_pts DESC
"""

raw_df = conn.sql(feature_sql).pl()
print(f"Feature matrix: {raw_df.shape[0]:,} player-seasons x {raw_df.shape[1]} columns")
print(f"Seasons covered: {sorted(raw_df['season_year'].unique().to_list())[:5]} ... {sorted(raw_df['season_year'].unique().to_list())[-5:]}")
raw_df.head(5)

In [ ]:
# --- Define feature groups and normalize ---

STAT_FEATURES = [
    "avg_pts", "avg_reb", "avg_ast", "avg_stl", "avg_blk", "avg_tov",
    "avg_ts_pct", "avg_usg_pct", "avg_net_rating", "avg_pie",
    "fg3_pct", "ft_pct",
]
STYLE_FEATURES = [
    "iso_poss_frac", "pnr_bh_frac", "spotup_frac",
    "postup_frac", "transition_frac", "cut_frac",
]
HUSTLE_FEATURES = [
    "contested_shots_pg", "deflections_pg", "loose_balls_pg",
    "charges_drawn_pg", "screen_assists_pg",
]
PHYSICAL_NUMERIC = ["height_in", "weight"]
PHYSICAL_CAT = ["position"]

ALL_NUMERIC = STAT_FEATURES + STYLE_FEATURES + HUSTLE_FEATURES + PHYSICAL_NUMERIC
ALL_FEATURES = ALL_NUMERIC + PHYSICAL_CAT

# Fill nulls with column median for numeric, 'Unknown' for categorical
df = raw_df.clone()
for col in ALL_NUMERIC:
    median_val = df[col].drop_nulls().median()
    df = df.with_columns(pl.col(col).fill_null(median_val))
df = df.with_columns(pl.col("position").fill_null("Unknown"))

# Normalize numeric features with StandardScaler
scaler = StandardScaler()
numeric_matrix = df.select(ALL_NUMERIC).to_numpy().astype(float)
scaled_matrix = scaler.fit_transform(numeric_matrix)

print(f"Scaled feature matrix shape: {scaled_matrix.shape}")
print(f"Feature groups: {len(STAT_FEATURES)} statistical, {len(STYLE_FEATURES)} play style, "
      f"{len(HUSTLE_FEATURES)} hustle, {len(PHYSICAL_NUMERIC)} physical numeric, "
      f"{len(PHYSICAL_CAT)} physical categorical")
print(f"Total numeric features: {len(ALL_NUMERIC)}")

# Create an index for quick lookups
player_labels = (
    df.select("player_name", "season_year")
    .with_columns(
        (pl.col("player_name") + " (" + pl.col("season_year") + ")").alias("label")
    )["label"].to_list()
)

takeaway(
    f"Built a {scaled_matrix.shape[0]:,} x {scaled_matrix.shape[1]} normalized feature matrix "
    f"spanning {df['season_year'].n_unique()} seasons. Each player-season is described by "
    f"26 numeric features across statistical, play style, hustle, and physical dimensions, "
    f"plus a categorical position feature for Gower distance."
)

<a id="3"></a>
## 3. Similarity Methods

We implement three complementary approaches to player similarity:

| Method | How it works | Strengths | Weaknesses |
|--------|-------------|-----------|------------|
| **Cosine Similarity** | Angle between feature vectors | Captures *shape* of profile regardless of magnitude | Ignores absolute differences |
| **Euclidean Distance** | Straight-line distance in feature space | Intuitive, respects magnitude | Sensitive to scale (needs normalization) |
| **Gower Distance** | Mixed-type metric (numeric + categorical) | Naturally handles position as categorical | Slower on large matrices |

In [ ]:
# --- Method 1: Cosine Similarity ---
cos_sim = cosine_similarity(scaled_matrix)

# --- Method 2: Euclidean Distance (via NearestNeighbors) ---
nn_euclidean = NearestNeighbors(n_neighbors=11, metric="euclidean")
nn_euclidean.fit(scaled_matrix)

# --- Method 3: Approximate Gower via NearestNeighbors ---
# Instead of O(N²) Gower matrix, use ball-tree with manhattan distance
# One-hot encode the categorical 'position' column
position_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
position_encoded = position_encoder.fit_transform(df[["position"]].to_pandas())

# Build combined matrix: scaled numeric + encoded position
gower_approx_matrix = np.hstack([scaled_matrix, position_encoded * 0.5])  # weight position at 0.5

# Replace full-matrix Gower with NearestNeighbors for top-K lookup
nn_gower = NearestNeighbors(n_neighbors=11, algorithm="ball_tree", metric="manhattan")
nn_gower.fit(gower_approx_matrix)

print(f"Approximate Gower NN model fitted ({gower_approx_matrix.shape[0]} players, {gower_approx_matrix.shape[1]} features)")


def get_top_similar(query_label, method="cosine", n=10):
    """Return top-N most similar players for a given query label."""
    try:
        idx = player_labels.index(query_label)
    except ValueError:
        raise ValueError(f"Player '{query_label}' not found in dataset")

    if method == "cosine":
        sims = cos_sim[idx]
        # Sort descending (higher = more similar), skip self
        ranked = np.argsort(sims)[::-1]
        ranked = [r for r in ranked if r != idx][:n]
        return [(player_labels[r], float(sims[r])) for r in ranked]

    elif method == "euclidean":
        distances, indices = nn_euclidean.kneighbors(scaled_matrix[idx:idx+1])
        results = []
        for d, i in zip(distances[0], indices[0]):
            if i != idx:
                results.append((player_labels[i], float(d)))
        return results[:n]

    elif method == "gower":
        distances, indices = nn_gower.kneighbors(gower_approx_matrix[idx:idx+1])
        results = []
        for d, i in zip(distances[0], indices[0]):
            if i != idx:
                results.append((player_labels[i], float(d)))
        return results[:n]

    else:
        raise ValueError(f"Unknown method: {method}")


print("All three similarity methods ready.")

In [ ]:
# --- Find a well-known query player (top scorer in the most recent season) ---

latest_season = df["season_year"].unique().sort(descending=True).to_list()[0]
top_scorer = (
    df.filter(pl.col("season_year") == latest_season)
    .sort("avg_pts", descending=True)
    .head(1)
)
query_name = top_scorer["player_name"][0]
query_label = f"{query_name} ({latest_season})"
print(f"Query player: {query_label} ({top_scorer['avg_pts'][0]:.1f} PPG)")

# Get top-10 by each method
cos_results = get_top_similar(query_label, "cosine", 10)
euc_results = get_top_similar(query_label, "euclidean", 10)
gow_results = get_top_similar(query_label, "gower", 10)

# Build comparison table
max_rows = 10
table_data = {
    "Rank": list(range(1, max_rows + 1)),
    "Cosine (similarity)": [f"{name}  [{score:.3f}]" for name, score in cos_results],
    "Euclidean (distance)": [f"{name}  [{score:.2f}]" for name, score in euc_results],
    "Gower (distance)": [f"{name}  [{score:.3f}]" for name, score in gow_results],
}

fig = go.Figure(data=[go.Table(
    header=dict(
        values=[f"<b>{col}</b>" for col in table_data.keys()],
        fill_color=COLORS["purple"],
        font=dict(color="white", size=13),
        align="center",
    ),
    cells=dict(
        values=list(table_data.values()),
        fill_color=[
            [COLORS["light"]] * max_rows,
            ["white"] * max_rows,
            ["white"] * max_rows,
            ["white"] * max_rows,
        ],
        font=dict(size=12),
        align=["center", "left", "left", "left"],
        height=28,
    ),
)])

fig.update_layout(
    title=f"Top-10 Most Similar Players to {query_label}",
    height=420,
    template=TEMPLATE,
    margin=dict(l=20, r=20, t=50, b=20),
)
fig.show()

# Find players that appear in all three top-10 lists
cos_names = {name for name, _ in cos_results}
euc_names = {name for name, _ in euc_results}
gow_names = {name for name, _ in gow_results}
consensus = cos_names & euc_names & gow_names

takeaway(
    f"The three methods agree on {len(consensus)} player(s) in the top 10: "
    f"{', '.join(sorted(consensus)) if consensus else 'none'}. "
    f"Cosine similarity favors players with similar *profiles* regardless of volume. "
    f"Euclidean distance penalizes absolute stat differences. "
    f"Gower distance naturally incorporates position as a categorical match, "
    f"making it most useful when comparing across positions."
)

<a id="4"></a>
## 4. Interactive Similarity Finder

Select any of the top-30 players (by minutes) in the current season from a dropdown
to see their radar chart overlay with their most similar player, plus a stats
comparison table.

> **Note:** This section uses **raw-scaled** cosine similarity — player stats are compared at face value without adjusting for era-specific trends (pace, 3-point revolution, etc.). See Section 5 for era-adjusted comparisons.

In [ ]:
# --- Helper: make a radar chart for two players ---

RADAR_DIMS = [
    ("avg_pts", "PTS"),
    ("avg_ast", "AST"),
    ("avg_reb", "REB"),
    ("avg_stl", "STL+BLK"),  # will combine
    ("avg_ts_pct", "TS%"),
    ("avg_usg_pct", "USG%"),
    ("fg3_pct", "3PT%"),
    ("avg_pie", "PIE"),
]
RADAR_LABELS = [label for _, label in RADAR_DIMS]


def get_radar_values(row_idx):
    """Compute percentile ranks for radar dimensions (0-100)."""
    values = []
    for col, label in RADAR_DIMS:
        if label == "STL+BLK":
            # Combine steals + blocks
            player_val = float(df[row_idx, "avg_stl"]) + float(df[row_idx, "avg_blk"])
            all_vals = (df["avg_stl"] + df["avg_blk"]).to_numpy()
        else:
            player_val = float(df[row_idx, col])
            all_vals = df[col].to_numpy()
        # Percentile rank
        pctile = (np.sum(all_vals <= player_val) / len(all_vals)) * 100
        values.append(pctile)
    return values


def make_radar(idx_query, idx_match, sim_score, fig=None, row=None, col=None):
    """Create a radar (Scatterpolar) overlay for two players."""
    q_vals = get_radar_values(idx_query)
    m_vals = get_radar_values(idx_match)

    # Close the polygon
    q_vals_closed = q_vals + [q_vals[0]]
    m_vals_closed = m_vals + [m_vals[0]]
    labels_closed = RADAR_LABELS + [RADAR_LABELS[0]]

    if fig is None:
        fig = go.Figure()

    fig.add_trace(go.Scatterpolar(
        r=q_vals_closed,
        theta=labels_closed,
        fill="toself",
        name=player_labels[idx_query],
        fillcolor=f"rgba(85, 37, 131, 0.2)",
        line=dict(color=COLORS["purple"], width=2),
    ))
    fig.add_trace(go.Scatterpolar(
        r=m_vals_closed,
        theta=labels_closed,
        fill="toself",
        name=f"{player_labels[idx_match]} (sim={sim_score:.3f})",
        fillcolor=f"rgba(253, 185, 39, 0.2)",
        line=dict(color=COLORS["gold"], width=2),
    ))
    return fig

In [ ]:
# --- Pre-compute similarity for top-30 players by minutes in current season ---

current_season_df = df.filter(pl.col("season_year") == latest_season)
top30 = (
    current_season_df
    .sort("total_min", descending=True)
    .head(30)
)

# For each top-30 player, find their most similar + top-10
dropdown_data = {}  # label -> {match_label, sim_score, query_idx, match_idx}
for row in top30.iter_rows(named=True):
    label = f"{row['player_name']} ({row['season_year']})"
    if label not in player_labels:
        continue
    idx = player_labels.index(label)
    sims = cos_sim[idx].copy()
    sims[idx] = -1  # exclude self
    best_idx = int(np.argmax(sims))
    dropdown_data[label] = {
        "match_label": player_labels[best_idx],
        "sim_score": float(sims[best_idx]),
        "query_idx": idx,
        "match_idx": best_idx,
    }

print(f"Pre-computed similarities for {len(dropdown_data)} players")

# Build the Plotly figure with updatemenus dropdown
sorted_labels = sorted(dropdown_data.keys())
default_label = sorted_labels[0]

# Create all traces (2 per player: query radar + match radar), initially invisible
fig = go.Figure()
visibility_map = {}  # label -> list of trace indices

for i, label in enumerate(sorted_labels):
    info = dropdown_data[label]
    q_vals = get_radar_values(info["query_idx"])
    m_vals = get_radar_values(info["match_idx"])
    q_closed = q_vals + [q_vals[0]]
    m_closed = m_vals + [m_vals[0]]
    labels_closed = RADAR_LABELS + [RADAR_LABELS[0]]

    visible = (label == default_label)
    trace_idx_start = len(fig.data)

    fig.add_trace(go.Scatterpolar(
        r=q_closed,
        theta=labels_closed,
        fill="toself",
        name=label,
        fillcolor="rgba(85, 37, 131, 0.2)",
        line=dict(color=COLORS["purple"], width=2),
        visible=visible,
    ))
    fig.add_trace(go.Scatterpolar(
        r=m_closed,
        theta=labels_closed,
        fill="toself",
        name=f"{info['match_label']} (sim={info['sim_score']:.3f})",
        fillcolor="rgba(253, 185, 39, 0.2)",
        line=dict(color=COLORS["gold"], width=2),
        visible=visible,
    ))
    visibility_map[label] = [trace_idx_start, trace_idx_start + 1]

# Build dropdown buttons
buttons = []
total_traces = len(fig.data)
for label in sorted_labels:
    vis = [False] * total_traces
    for idx in visibility_map[label]:
        vis[idx] = True
    info = dropdown_data[label]
    buttons.append(dict(
        label=label.split(" (")[0],  # short name
        method="update",
        args=[
            {"visible": vis},
            {"title": f"Player Twin: {label} vs {info['match_label']}"}
        ],
    ))

fig.update_layout(
    template=TEMPLATE,
    polar=dict(
        radialaxis=dict(range=[0, 100], ticksuffix="%", showticklabels=True),
    ),
    title=f"Player Twin: {default_label} vs {dropdown_data[default_label]['match_label']}",
    height=650,
    showlegend=True,
    updatemenus=[
        dict(
            active=0,
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.0,
            xanchor="left",
            y=1.15,
            yanchor="top",
            bgcolor="white",
            bordercolor=COLORS["purple"],
            font=dict(size=12),
        )
    ],
    annotations=[
        dict(
            text="Select player:",
            x=0.0, y=1.20, xref="paper", yref="paper",
            align="left", showarrow=False,
            font=dict(size=13, color=COLORS["purple"]),
        )
    ],
)
fig.show()

takeaway(
    f"Use the dropdown to explore player twins for any of the top 30 players by "
    f"minutes this season. The radar overlay shows percentile ranks across 8 "
    f"dimensions -- where the shapes overlap most tightly, the players are most "
    f"similar. Note how the engine finds matches across eras and teams, surfacing "
    f"connections that pure stat comparisons miss."
)

<a id="5"></a>
## 5. Cross-Era Comparison

Raw stats are misleading across eras -- pace, rules, and three-point volume
have changed dramatically. We **era-adjust** by z-scoring each feature within
its season, then find historical twins for current stars.

> **Note:** This section uses **era-adjusted** cosine similarity — each stat is z-scored within its season to compare players relative to their contemporaries, eliminating era-specific inflation.

In [ ]:
# --- Era-adjust features: z-score within each season ---

era_df = df.clone()

# Exclude seasons with too few players for meaningful z-scoring
season_counts = era_df.group_by("season_year").len()
valid_seasons = season_counts.filter(pl.col("len") >= 5)["season_year"]
era_df = era_df.filter(pl.col("season_year").is_in(valid_seasons))

for col in ALL_NUMERIC:
    era_df = era_df.with_columns(
        ((pl.col(col) - pl.col(col).mean().over("season_year"))
         / pl.col(col).std().over("season_year"))
        .fill_nan(None)  # NaN from zero-std seasons → null (will be handled below)
        .fill_null(0.0)
        .alias(f"{col}_era")
    )

era_cols = [f"{col}_era" for col in ALL_NUMERIC]
era_matrix = era_df.select(era_cols).to_numpy().astype(float)

# Compute cosine similarity on era-adjusted features
era_cos_sim = cosine_similarity(era_matrix)

print(f"Era-adjusted matrix shape: {era_matrix.shape}")
print("Era-adjusted features subtract season mean and divide by season std,")
print("making a 20 PPG scorer in 2005 comparable to a 25 PPG scorer in 2024.")

In [ ]:
# --- Find historical twins for 5 current stars ---

# Pick 5 current stars: top 5 in PIE in the latest season
current_stars = (
    era_df.filter(pl.col("season_year") == latest_season)
    .sort("avg_pie", descending=True)
    .head(5)
)

# For each, find best historical match (different season, different player)
era_labels = era_df.select(
    (pl.col("player_name") + " (" + pl.col("season_year") + ")").alias("label")
)["label"].to_list()
era_seasons = era_df["season_year"].to_list()
era_player_ids = era_df["player_id"].to_list()  # build once for same-player exclusion

cross_era_results = []
for row in current_stars.iter_rows(named=True):
    label = f"{row['player_name']} ({row['season_year']})"
    if label not in era_labels:
        continue
    idx = era_labels.index(label)
    sims = era_cos_sim[idx].copy()
    sims[idx] = -1  # exclude self

    # Exclude same-season matches AND same-player matches (NB-031)
    for j in range(len(sims)):
        if era_seasons[j] == row["season_year"] or era_player_ids[j] == row["player_id"]:
            sims[j] = -1

    best_idx = int(np.argmax(sims))
    best_row = era_df[best_idx]

    cross_era_results.append({
        "Current Player": label,
        "PPG": f"{row['avg_pts']:.1f}",
        "RPG": f"{row['avg_reb']:.1f}",
        "APG": f"{row['avg_ast']:.1f}",
        "PIE": f"{row['avg_pie']:.3f}",
        "Historical Twin": era_labels[best_idx],
        "Twin PPG": f"{float(best_row['avg_pts'][0]):.1f}",
        "Twin RPG": f"{float(best_row['avg_reb'][0]):.1f}",
        "Twin APG": f"{float(best_row['avg_ast'][0]):.1f}",
        "Twin PIE": f"{float(best_row['avg_pie'][0]):.3f}",
        "Similarity": f"{float(sims[best_idx]):.3f}",
    })

# Display as Plotly table
if cross_era_results:
    cols = list(cross_era_results[0].keys())
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=[f"<b>{c}</b>" for c in cols],
            fill_color=COLORS["purple"],
            font=dict(color="white", size=12),
            align="center",
        ),
        cells=dict(
            values=[[r[c] for r in cross_era_results] for c in cols],
            fill_color=[
                [COLORS["light"]] * len(cross_era_results)
                if i < 5 else
                ["#e8f4e8"] * len(cross_era_results)
                for i in range(len(cols))
            ],
            font=dict(size=11),
            align="center",
            height=28,
        ),
    )])

    fig.update_layout(
        title="Cross-Era Player Twins (Era-Adjusted Cosine Similarity)",
        height=280,
        template=TEMPLATE,
        margin=dict(l=20, r=20, t=50, b=20),
    )
    fig.show()

    takeaway(
        "Era-adjusted similarity removes the inflation effect of modern pace and scoring. "
        "A player who dominates their era's average by 2 standard deviations is matched "
        "with a historical player who similarly dominated theirs. The raw stats may differ "
        "significantly, but the *relative dominance* and stylistic profile are nearly identical. "
        "This reveals true stylistic lineages that transcend rule changes."
    )
else:
    print("No cross-era results found (insufficient data overlap).")

<a id="6"></a>
## 6. Similarity Network

We compute the top-3 most similar players for everyone with >= 1000 minutes
in the latest season and visualize the resulting graph. Clusters reveal
player archetypes; isolated nodes are unique players with no close comp.

In [ ]:
import networkx as nx

# --- Build similarity network ---

# Filter to latest season, >= 1000 minutes
network_df = df.filter(
    (pl.col("season_year") == latest_season)
    & (pl.col("total_min") >= 1000)
)

# Get indices into the full matrix
network_indices = []
network_labels_short = []
network_positions = []
for row in network_df.iter_rows(named=True):
    label = f"{row['player_name']} ({row['season_year']})"
    if label in player_labels:
        network_indices.append(player_labels.index(label))
        network_labels_short.append(row["player_name"])
        network_positions.append(row["position"] or "Unknown")

print(f"Network nodes: {len(network_indices)} players with >= 1000 min in {latest_season}")

# Build graph: for each node, add edges to top-3 most similar within this group
G = nx.Graph()
for i, global_idx in enumerate(network_indices):
    G.add_node(i, name=network_labels_short[i], position=network_positions[i])

# Compute similarity sub-matrix for this group
sub_sim = cos_sim[np.ix_(network_indices, network_indices)]

for i in range(len(network_indices)):
    row_sims = sub_sim[i].copy()
    row_sims[i] = -1  # exclude self
    top3_local = np.argsort(row_sims)[-3:][::-1]
    for j in top3_local:
        if j != i:
            G.add_edge(i, j, weight=float(row_sims[j]))

print(f"Network edges: {G.number_of_edges()}")

In [ ]:
# --- Visualize with spring layout ---

pos = nx.spring_layout(G, seed=42, k=2.0 / np.sqrt(len(G.nodes())))

# Position color map
POS_COLORS = {
    "Guard": COLORS["blue"],
    "G": COLORS["blue"],
    "Forward": COLORS["green"],
    "F": COLORS["green"],
    "Center": COLORS["red"],
    "C": COLORS["red"],
    "Guard-Forward": COLORS["gold"],
    "G-F": COLORS["gold"],
    "Forward-Guard": COLORS["gold"],
    "F-G": COLORS["gold"],
    "Forward-Center": COLORS["orange"],
    "F-C": COLORS["orange"],
    "Center-Forward": COLORS["orange"],
    "C-F": COLORS["orange"],
}

fig = go.Figure()

# Draw edges
for u, v, data in G.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    weight = data.get("weight", 0.5)
    fig.add_trace(go.Scatter(
        x=[x0, x1, None],
        y=[y0, y1, None],
        mode="lines",
        line=dict(color="rgba(180,180,180,0.3)", width=max(0.5, weight * 2)),
        showlegend=False,
        hoverinfo="skip",
    ))

# Group nodes by position for legend
pos_groups = {}
for node_id in G.nodes():
    position = G.nodes[node_id]["position"]
    if position not in pos_groups:
        pos_groups[position] = {"x": [], "y": [], "text": [], "names": []}
    x, y = pos[node_id]
    # Build hover text with top-3 connections
    neighbors = []
    for neighbor in G.neighbors(node_id):
        edge_data = G.edges[node_id, neighbor]
        neighbors.append(
            f"{G.nodes[neighbor]['name']}: {edge_data['weight']:.3f}"
        )
    hover = (
        f"<b>{G.nodes[node_id]['name']}</b><br>"
        f"Position: {position}<br>"
        f"<br>Most similar:<br>" + "<br>".join(neighbors[:3])
    )
    pos_groups[position]["x"].append(x)
    pos_groups[position]["y"].append(y)
    pos_groups[position]["text"].append(hover)
    pos_groups[position]["names"].append(G.nodes[node_id]["name"])

for position, data in sorted(pos_groups.items()):
    color = POS_COLORS.get(position, COLORS["gray"])
    fig.add_trace(go.Scatter(
        x=data["x"],
        y=data["y"],
        mode="markers+text",
        marker=dict(color=color, size=12, line=dict(color="white", width=1)),
        text=[n.split(" ")[-1] for n in data["names"]],  # last name only
        textposition="top center",
        textfont=dict(size=9),
        name=position,
        hovertemplate="%{customdata}<extra></extra>",
        customdata=data["text"],
    ))

fig.update_layout(
    template=TEMPLATE,
    title=f"Player Similarity Network ({latest_season}, >= 1000 min)",
    height=750,
    width=900,
    showlegend=True,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
    legend=dict(title="Position"),
)
fig.show()

# Find most connected nodes (highest degree)
degrees = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:5]
most_connected = [(G.nodes[n]["name"], d) for n, d in degrees]

takeaway(
    f"The network reveals natural player clusters: guards connect to guards, "
    f"bigs connect to bigs, but some cross-position connections emerge for versatile "
    f"players. The most connected nodes are "
    + ", ".join(f"{name} ({d} connections)" for name, d in most_connected[:3])
    + " -- these are the archetypes around which other players cluster."
)

<a id="7"></a>
## 7. Draft Prospect Comparison

For recent draft picks, we combine their pre-draft **combine measurements**
(wingspan, vertical leap, agility, sprint) with their **first-season stats**
to find the most similar established player. This suggests potential career
trajectories.

In [ ]:
# --- Build draft prospect + early career feature vectors ---

draft_prospect_sql = f"""
WITH recent_picks AS (
    SELECT
        d.person_id,
        d.player_name,
        d.season AS draft_season,
        d.overall_pick,
        d.wingspan,
        d.height_wo_shoes,
        d.weight AS combine_weight,
        d.standing_vertical_leap,
        d.lane_agility_time,
        d.three_quarter_sprint
    FROM fact_draft d
    WHERE d.overall_pick IS NOT NULL
      AND d.overall_pick <= 60
      AND d.wingspan IS NOT NULL
),

first_season AS (
    SELECT
        aps.player_id,
        aps.season_year,
        aps.gp,
        aps.avg_pts AS rookie_ppg,
        aps.avg_reb AS rookie_rpg,
        aps.avg_ast AS rookie_apg,
        aps.avg_ts_pct AS rookie_ts,
        aps.avg_pie AS rookie_pie,
        ROW_NUMBER() OVER (
            PARTITION BY aps.player_id
            ORDER BY aps.season_year
        ) AS season_rank
    FROM agg_player_season aps
    WHERE aps.season_type = 'Regular Season'
      AND aps.gp >= 10
),

career AS (
    SELECT
        player_id,
        career_ppg,
        career_rpg,
        career_apg,
        seasons_played,
        career_gp
    FROM agg_player_career
)

SELECT
    rp.person_id,
    rp.player_name,
    rp.draft_season,
    rp.overall_pick,
    rp.wingspan,
    rp.height_wo_shoes,
    rp.combine_weight,
    rp.standing_vertical_leap,
    rp.lane_agility_time,
    rp.three_quarter_sprint,
    fs.rookie_ppg,
    fs.rookie_rpg,
    fs.rookie_apg,
    fs.rookie_ts,
    fs.rookie_pie,
    c.career_ppg,
    c.career_rpg,
    c.career_apg,
    c.seasons_played,
    c.career_gp
FROM recent_picks rp
JOIN first_season fs
    ON rp.person_id = fs.player_id AND fs.season_rank = 1
LEFT JOIN career c
    ON rp.person_id = c.player_id
ORDER BY rp.draft_season DESC, rp.overall_pick
"""

draft_df = conn.sql(draft_prospect_sql).pl()
print(f"Draft prospects with combine + rookie data: {len(draft_df):,}")
print(f"Draft seasons: {sorted(draft_df['draft_season'].unique().to_list())[-5:]}")
draft_df.head(5)

In [ ]:
# --- Find most similar established player for recent draft picks ---

# Combine features: physical + early stats
PROSPECT_FEATURES = [
    "wingspan", "height_wo_shoes", "combine_weight",
    "standing_vertical_leap", "lane_agility_time", "three_quarter_sprint",
    "rookie_ppg", "rookie_rpg", "rookie_apg", "rookie_ts", "rookie_pie",
]

# Filter to rows with all features present
prospect_df = draft_df.drop_nulls(subset=PROSPECT_FEATURES)

# Split into recent (last 3 draft seasons) and established (4+ seasons played)
all_seasons = sorted(prospect_df["draft_season"].unique().to_list())
recent_cutoff = all_seasons[-3] if len(all_seasons) >= 3 else all_seasons[0]

recent = prospect_df.filter(pl.col("draft_season") >= recent_cutoff)
established = prospect_df.filter(
    (pl.col("draft_season") < recent_cutoff)
    & (pl.col("seasons_played") >= 4)
)

print(f"Recent prospects (since {recent_cutoff}): {len(recent)}")
print(f"Established comparisons (4+ seasons): {len(established)}")

if len(recent) > 0 and len(established) > 0:
    # Normalize features
    all_prospect_data = pl.concat([recent, established])
    prospect_scaler = StandardScaler()
    all_prospect_matrix = prospect_scaler.fit_transform(
        all_prospect_data.select(PROSPECT_FEATURES).to_numpy().astype(float)
    )

    n_recent = len(recent)
    recent_matrix = all_prospect_matrix[:n_recent]
    established_matrix = all_prospect_matrix[n_recent:]

    # Cosine similarity between recent and established
    prospect_sim = cosine_similarity(recent_matrix, established_matrix)

    # For each recent pick, find best established match
    comp_results = []
    for i in range(min(n_recent, 20)):  # limit to top 20 for display
        best_j = int(np.argmax(prospect_sim[i]))
        sim_score = float(prospect_sim[i][best_j])
        r = recent[i]
        e = established[best_j]

        comp_results.append({
            "Draft Pick": f"{r['player_name'][0]} (#{r['overall_pick'][0]}, {r['draft_season'][0]})",
            "Rookie PPG": f"{float(r['rookie_ppg'][0]):.1f}",
            "Similar To": f"{e['player_name'][0]} (#{e['overall_pick'][0]}, {e['draft_season'][0]})",
            "Comp Rookie PPG": f"{float(e['rookie_ppg'][0]):.1f}",
            "Comp Career PPG": f"{float(e['career_ppg'][0]):.1f}" if e['career_ppg'][0] is not None else "N/A",
            "Comp Seasons": str(int(e['seasons_played'][0])) if e['seasons_played'][0] is not None else "N/A",
            "Similarity": f"{sim_score:.3f}",
        })

    if comp_results:
        cols = list(comp_results[0].keys())
        fig = go.Figure(data=[go.Table(
            header=dict(
                values=[f"<b>{c}</b>" for c in cols],
                fill_color=COLORS["purple"],
                font=dict(color="white", size=12),
                align="center",
            ),
            cells=dict(
                values=[[r[c] for r in comp_results] for c in cols],
                fill_color=[
                    [COLORS["light"]] * len(comp_results),
                    ["white"] * len(comp_results),
                    ["#e8f4e8"] * len(comp_results),
                    ["white"] * len(comp_results),
                    ["white"] * len(comp_results),
                    ["white"] * len(comp_results),
                    ["white"] * len(comp_results),
                ],
                font=dict(size=11),
                align="center",
                height=28,
            ),
        )])

        fig.update_layout(
            title=f"Draft Prospect Comparisons: Combine + Rookie Stats",
            height=min(800, 80 + 30 * len(comp_results)),
            template=TEMPLATE,
            margin=dict(l=20, r=20, t=50, b=20),
        )
        fig.show()

        takeaway(
            "By combining pre-draft physical measurements with first-season production, "
            "we can find the historical player whose early career most resembles each "
            "recent draft pick. The comparison player's full career trajectory offers "
            "a data-driven projection -- though development, injury, and team context "
            "mean these are starting points for analysis, not destiny."
        )
else:
    print("Insufficient data to build prospect comparisons.")

<a id="8"></a>
## 8. Conclusion & Cross-Links

In [ ]:
# Cleanup handled by nbadb_utils.get_connection() via atexit
print("Analysis complete.")

### Summary

This notebook built a **multi-dimensional player similarity engine** using three
complementary methods across four feature categories unique to this dataset:

1. **Feature Engineering**: 26 numeric + 1 categorical feature per player-season,
   spanning box scores, Synergy play types, hustle stats, and physical profiles.
2. **Three Similarity Methods**: Cosine (profile shape), Euclidean (absolute distance),
   and Gower (mixed-type) each surface different player connections.
3. **Interactive Finder**: Dropdown-driven radar chart to explore any top player's twin.
4. **Cross-Era Comparison**: Z-scored within-season features enable fair comparisons
   across decades of rule and pace changes.
5. **Similarity Network**: Graph visualization reveals natural player archetypes
   and cross-position versatility.
6. **Draft Prospect Comps**: Combine measurements + rookie stats projected onto
   established career trajectories.

---

### nbadb Notebook Suite

| Part | Notebook | Description |
|---|---|---|
| Part 1 | [MVP Prediction](./nba_mvp_predictor.ipynb) | MVP Prediction with Tracking & Synergy Data |
| Part 2 | [Data-Driven Player Archetypes](./nba_player_archetypes.ipynb) | Data-Driven Player Archetypes (UMAP + GMM) |
| Part 3 | [Game Outcome Prediction](./nba_game_prediction.ipynb) | Game Outcome Prediction (Stacking Ensemble) |
| Part 4 | [Draft Combine to Career](./nba_draft_combine_analysis.ipynb) | Draft Combine to Career Prediction |
| Part 5 | [Defense Decoded](./nba_defense_decoded.ipynb) | Defense Decoded (Tracking + Hustle + Synergy PCA) |
| Part 6 | [Interactive Player Explorer](./nba_player_dashboard.ipynb) | Interactive Player Explorer |
| Part 7 | [Spatial Shot Analysis](./nba_shot_chart_analysis.ipynb) | Spatial Shot Analysis & 3-Point Revolution |
| **Part 8** | **Player Similarity** (this notebook) | **Player Similarity Engine (Cosine + Manhattan)** |
| Part 9 | [Career Trajectory](./nba_aging_curves.ipynb) | Career Trajectory & Aging Curve Modeling |
| Part 10 | [Play-by-Play Insights](./nba_play_by_play_insights.ipynb) | Play-by-Play: Win Probability, Runs & Clutch |

---

**Dataset**: [wyattowalsh/basketball on Kaggle](https://www.kaggle.com/datasets/wyattowalsh/basketball)
**Documentation**: [nbadb.dev](https://nbadb.dev)
**Source**: [github.com/wyattowalsh/nbadb](https://github.com/wyattowalsh/nbadb)